In [1]:
import pandas as pd

## Base vs RAG correctness

In [3]:
for method in ["base", 'rag']:
    df = pd.read_csv(f"res-{method}-judge.csv")
    correct = len(df[df['judge_output'] == 'Yes'])
    total = len(df)
    percentage_correct = (correct * 100)/total
    print(f'{method.upper()}, correct = {correct}/{total} ({percentage_correct:.2f}%)')

BASE, correct = 435/918 (47.39%)
RAG, correct = 736/918 (80.17%)


## Error Analysis

In [15]:
df = pd.read_csv('res-rag-judge.csv')
incorrect = df[df['judge_output'] == 'No']

In [50]:
error_df = incorrect.sample(50, random_state=42)

In [31]:
from utils.chroma import init_chroma_client, init_collection, query_collection
from IPython.display import clear_output

In [6]:
client = init_chroma_client()
collection = init_collection(client)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
errors = []
docs_str = []
rows = list(error_df.iterrows())

for i, item in enumerate(rows):
    clear_output(wait=True)
    row = item[-1]
    query_result = query_collection(collection, row['question'])
    docs = query_result["documents"][0]
    
    print(f"\nROW {i+1} of {len(error_df)}")
    print(f"Question: {row['question']}")
    print(f"Answer: {row['answer']}")
    print(f"Prediction: {row['prediction']}")
    print(f"\nDocs:")
    for j, doc in enumerate(docs):
        print(f"{j+1}. {doc}")
        
    print()
    
    # Get input
    note = input("Error Type (Generation, Retrieval, Annotation, Judgement, Vague question): ")
    errors.append(note)
    docs_str.append(str(docs))

# Save results to csv
output_file = 'error_annotations.csv'
error_df['error_type'] = errors
error_df['docs'] = docs_str
error_df.to_csv(output_file, index=False)


ROW 50 of 50
Question: hard
Answer: hard
Prediction: The provided context is insufficient to answer the question.

Docs:
1. |
2.  ;
3. |-
4. |-
5. |}



Error Type (Generation, Retrieval, Annotation, Judgement, Vague question):  Vague question


In [53]:
error_df['error_type'].value_counts()

error_type
Retrieval         13
Generation        13
Annotation        10
Vague question     9
Judgement          5
Name: count, dtype: int64

## RAG vs RAG-Reranker correctness

In [5]:
for method in ["rag", 'rag-reranked']:
    df = pd.read_csv(f"res-{method}-judge.csv")
    correct = len(df[df['judge_output'] == 'Yes'])
    total = len(df)
    percentage_correct = (correct * 100)/total
    print(f'{method.upper()}, correct = {correct}/{total} ({percentage_correct:.2f}%)')

RAG, correct = 736/918 (80.17%)
RAG-RERANKED, correct = 734/918 (79.96%)
